# H2O Implementation -- official H2O engine (submodule) + KVQuant-family harness -- 40% budget (GSM8K + ARC-Challenge + HellaSwag)

This notebook evaluates H2O using the **official FMInference/H2O code from the
`H2O` submodule of the KVCacheCompression repo** -- specifically
`h2o_hf/utils_real_drop/modify_llama.py`'s `H2OKVCache_LayerWise`, the
authors' real-KV-dropping eviction engine (per-layer AND per-head heavy-hitter
scores, physical pruning of the cache tensors) -- inside the exact same
experimental harness as the KVQuant-family notebooks
(`KVQuant_2bit/3bit/4bit_Implementation.ipynb`,
`KVQuant_Baseline_Implementation.ipynb`): same model, same pinned package
versions, same seed, same dataset loading/tokenization, same GSM8K
prompt + question set + answer grading, same metric definitions, same
result-CSV schema.

This is one of three H2O eviction-budget variants (40% / 40% / 60% total
cache budget) -- identical in every other respect to the 40% and 60% budget
notebooks. Only `HEAVY_RATIO` / `RECENT_RATIO` and `METHOD_NAME` differ
between the three.

Full-precision-baseline comparison is over three datasets: GSM8K (generative,
extract the final number), ARC-Challenge, and HellaSwag (both multiple-choice,
scored generate-and-extract: the model generates and we parse the chosen
letter).

All three datasets share the SAME real generate/decode measurement path (the
official H2O engine evicting inside the timed loop, same as everywhere in
this notebook), so TTFT/TBT/latency, KV-cache memory (MEASURED from the real,
physically pruned cache), and per-answer perplexity are directly comparable
across datasets. This is deliberate: because the point of the notebook is to
measure generation timing and KV-cache memory, the MCQ datasets use
generate-and-extract rather than the leaderboard log-likelihood (`acc_norm`)
protocol -- a single teacher-forced forward pass would not exercise the
decode-time eviction the way generation does. Consequently the MCQ accuracies
here are NOT directly comparable to Open-LLM-Leaderboard HellaSwag/ARC
numbers.

Few-shot prefixes for ARC and HellaSwag are NOT hand-written. Following the
lm-eval-harness convention, exemplars are drawn by a seeded `random.Random`
sample from each dataset's own train split (non-CoT, direct-answer format:
question + lettered options + `Answer: <letter>`). The eval pool for each
dataset is the *remainder* of its train split (train minus the sampled
few-shot exemplars) combined with its full test (ARC) / validation
(HellaSwag) split; the reproducible random eval subset is drawn from that
combined pool. No exemplar ever also appears as an eval question, so there is
no leakage. ARC uses 25-shot and HellaSwag 10-shot by default (the
leaderboard conventions); both are configurable constants. HellaSwag
context/endings get the standard harness text cleanup before formatting.

**What comes from the submodule vs. what is harness code:**

- FROM THE SUBMODULE (unmodified): `H2OKVCache_LayerWise` -- heavy-hitter
  score accumulation (`_update_hh_score`: attention mass summed over batch
  and query rows, accumulated per head per cached position), the keep-set
  selection (per-head top-`hh_size` heavy hitters + last `recent_size`
  recent tokens), and the physical KV dropping. One engine instance per
  layer, exactly as `H2OLlamaAttention` instantiates it in the official
  code.
- HARNESS CODE (this notebook): the loop that runs the stock Llama-3.1-8B with
  `output_attentions=True` and feeds each layer's attention weights + legacy
  KV tuple through that layer's engine after every forward. The submodule's
  full `H2OLlamaAttention` module replacement could not be used directly
  because it targets an older transformers API (no `cache_position`, legacy
  tuple plumbing) and **assumes MHA** -- its cache-vs-score shapes break on
  GQA models like Llama-3.1-8B (8 KV heads vs. 32 query heads). Under the
  pinned `transformers==4.43.4` with eager attention, the stock model
  computes the identical attention math that class reimplements, so driving
  the official engine from the stock model's attention outputs preserves the
  method exactly.
- ONE GQA ADAPTATION (in harness code, submodule untouched): the engine
  expects one score row per cached KV head. Llama-3.1-8B's 32 query heads share
  8 KV heads (groups of 4), so each layer's attention weights are reduced to
  KV heads by summing the 4 query heads in each group -- the total attention
  mass received by that KV head's cache entries, the natural GQA
  generalization of the paper's per-head scores.

**Method requirements that legitimately differ from the KVQuant notebooks:**

1. `attn_implementation="eager"` -- H2O needs real attention weights;
   FlashAttention/SDPA cannot return them. The full-precision baseline
   notebook also now runs eager with the same hand-rolled prefill/decode
   loop structure (see its Setup section), so there is no longer an
   "eager tax" confound between this notebook and the baseline; any
   remaining latency gap is attributable to H2O's eviction overhead.
2. Memory here is **measured** (the engine physically drops KV positions);
   the quantized KVQuant notebooks report **calculated** bytes (simulated
   compression). Keep that distinction in mind when comparing memory columns.

**Budgets:** the official engine takes absolute `hh_size`/`recent_size`
counts (as in the repo's `run_summarization.py`). They are derived per sample
from ratios: `hh_size = int(0.10 * total_len)`, `recent_size = int(0.10 *
total_len)` -- i.e. a **40% total budget** (half of the other two H2O
notebooks' 40% and 60%), evenly split between the heavy-hitter and recency
components, matching that same even-split convention. `total_len` is always
this sample's own PROMPT length, known at prefill time, for all three
datasets including GSM8K -- there is no lookahead into how many tokens will
be generated. This is a deliberate design choice, not a negligible one: on
GSM8K, up to `GSM8K_MAX_NEW_TOKENS` (256) tokens are generated against a
budget sized before any of them existed, so a long generation decodes
against a cache budget that never accounted for its own length. Fresh
engines (fresh scores) are created per sample, mirroring the official
per-request `_clean_scores` usage.

**Harness parity (identical to the KVQuant family):** GSM8K, ARC-Challenge,
and HellaSwag are fully loaded and preprocessed first, then up to **2,048
random valid examples per dataset** are chosen with seed **42** (ARC and
HellaSwag draw that subset from the combined leftover-train + test/validation
pool described above). Every method therefore receives the same reproducible
subset. Selected indices are sorted into source order before evaluation,
avoiding an arbitrary shuffled execution order while retaining a genuinely
random subset. All prompting, scoring, latency, perplexity, accuracy, memory
aggregation, and CSV-output formulas remain unchanged. On every dataset,
including GSM8K, the reported H2O peak (`peak_memory_mb`) is always the
POST-eviction cache size -- the transient, budget-independent moment that
exists right after a batched prefill forward but before that step's
eviction pass has run is never counted (see `generate_gsm8k_h2o`'s and
`generate_mcq_h2o`'s header comments). There is no separate pre-eviction or
"steady-state" figure printed anywhere in this notebook.

Run cells top to bottom. Needs a GPU runtime. Before comparing across
notebooks, confirm the printed GPU name matches the other runs.

## Setup

In [ ]:
!hostname

In [ ]:
# Block 1 - Environment setup
# Run once per fresh runtime. Package versions are pinned so environment
# differences are never a confound between compression methods -- kept
# byte-for-byte identical to the KVQuant-family notebooks (including
# datasets==2.14.5, which the previous H2O notebook left unpinned).

from google.colab import drive
drive.mount("/content/drive")

!python -m pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.20.3" \
  "huggingface_hub==0.36.2" \
  sentencepiece \
  einops

!python -m pip install -q \
  "datasets==2.14.5" \
  tqdm \
  matplotlib

!python -m pip install -q --no-deps --force-reinstall "huggingface_hub==0.36.2"

import os

# Patch - transformers==4.43.4 hard-enforces "tokenizers>=0.19,<0.20" at
# IMPORT time (transformers/dependency_versions_check.py), not just at
# pip-install time -- so even though tokenizers==0.20.3 installs fine
# (needed since 0.19.x has no Python 3.13 wheel), "import transformers"
# still raises ImportError unless this hardcoded constraint is relaxed on
# disk first. Patched via a direct file edit -- importing transformers to
# patch it in-memory is not an option, since that import is exactly what
# triggers the failing check. Safe to run even on the old (working)
# environment: on Python <3.13 this is a no-op the moment the constraint
# line has already been relaxed, and re-running it is idempotent.
import importlib.util
import re

_transformers_spec = importlib.util.find_spec("transformers")
_deps_table_path = os.path.join(os.path.dirname(_transformers_spec.origin), "dependency_versions_table.py")

with open(_deps_table_path, "r") as f:
    _deps_content = f.read()

_deps_content_new, _n_subs = re.subn(
    r'("tokenizers":\s*)"tokenizers>=0\.19,<0\.20"',
    r'\1"tokenizers>=0.19,<0.21"',
    _deps_content,
)
if _n_subs > 0:
    with open(_deps_table_path, "w") as f:
        f.write(_deps_content_new)
    print(f"Patched {_deps_table_path}: tokenizers constraint relaxed to <0.21 "
          "(allows tokenizers==0.20.3 -- 0.19.x has no Python 3.13 wheel).")
else:
    print(f"NOTE: tokenizers constraint in {_deps_table_path} was not the expected "
          "'>=0.19,<0.20' (already patched, or transformers version differs) -- "
          "no change made.")

try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- Llama-3.1-8B is GATED: this will fail to load without a token that has accepted the Meta license at https://huggingface.co/meta-llama/Llama-3.1-8B")

print("Block 1 finished. Now run Block 2.")

In [ ]:
# Block 2 - Imports, GPU check

import gc
import math
import os
import re
import shutil
import time
import random
import pickle
import sys

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import datasets
import transformers
import huggingface_hub
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
MODEL_DTYPE = torch.bfloat16 if HAS_CUDA else torch.float32


def clear_hf_dataset_cache(*dataset_names):
    """Removes cached files for the given HF dataset repo name(s) (e.g.
    "wikitext", "gsm8k") from both the datasets cache and the hub cache.
    Used as an on_retry hook: a download that breaks partway through can
    leave a corrupted partial file that every subsequent retry just
    resumes (and re-breaks at the same point) instead of truly restarting
    -- clearing the cache forces a genuinely fresh download."""
    home = os.path.expanduser("~")
    for name in dataset_names:
        for base in [
            os.path.join(home, ".cache", "huggingface", "datasets", name),
            os.path.join(home, ".cache", "huggingface", "hub", f"datasets--{name}"),
        ]:
            shutil.rmtree(base, ignore_errors=True)


if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be very slow.")

# NOTE: clear_hf_dataset_cache is defined here (not in Helper Functions,
# below) for consistency with the KVQuant-family notebooks, where a
# calibration step needs it available early in Setup. This notebook has no
# such dependency itself, but keeping the split identical across the whole
# notebook family avoids Helper Functions containing different things in
# different notebooks. robust_call/sync_if_cuda/clear_memory have no early
# dependency anywhere and live in Helper Functions with the rest of the
# genuinely cross-dataset machinery.

In [ ]:
# Block 3 - Experiment settings.
# Sampling policy for GSM8K, ARC-Challenge, and HellaSwag (the only datasets
# in this notebook): up to 2,048 random valid examples each, deterministic
# seed 42. Sampling happens after validity filtering; selected indices are
# sorted back into source order so the subset is random while evaluation
# order stays stable. seeded_subset/robust_call now live in Helper Functions
# (below) rather than here -- see that section's header note.

LOCAL_MODEL_PATH = "/content/llama-3.1-8b"
HF_MODEL_ID = "meta-llama/Llama-3.1-8B"
MODEL_ID = LOCAL_MODEL_PATH if os.path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 42
QA_EVAL_SAMPLES = 2048
GSM8K_MAX_NEW_TOKENS = 256
METHOD_NAME = "h2o_budget_40pct"

# ---- H2O method hyperparameters (the only method-specific settings) ----
# The official engine receives absolute heavy-hitter and recent-token counts,
# derived per sample from the ratios below.
HEAVY_RATIO = 0.2
RECENT_RATIO = 0.2


random.seed(SHARED_SEED)
np.random.seed(SHARED_SEED)
torch.manual_seed(SHARED_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SHARED_SEED)

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation step by step, then end with exactly this format:\n"
    "#### <final number>\n\n"

    "Question: There are 15 trees in the grove. Grove workers will plant trees today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\n"
    "Answer: There are 15 trees originally. After planting, there are 21 trees. So the workers planted 21 - 15 = 6 trees.\n"
    "#### 6\n\n"

    "Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\n"
    "Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5 cars.\n"
    "#### 5\n\n"

    "Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\n"
    "Answer: Leah and her sister started with 32 + 42 = 74 chocolates. After eating 35, they have 74 - 35 = 39 left.\n"
    "#### 39\n\n"

    "Question: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\n"
    "Answer: Jason started with 20 lollipops and now has 12. So he gave away 20 - 12 = 8 lollipops.\n"
    "#### 8\n\n"

    "Question: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?\n"
    "Answer: Shawn started with 5 toys. He got 2 from mom and 2 from dad, which is 2 + 2 = 4 more toys. 5 + 4 = 9 toys total.\n"
    "#### 9\n\n"

    "Question: There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?\n"
    "Answer: 4 days from Monday to Thursday, with 5 computers installed each day, is 4 * 5 = 20 computers added. 9 + 20 = 29 computers total.\n"
    "#### 29\n\n"

    "Question: Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?\n"
    "Answer: Michael started with 58 golf balls. After losing 23 on Tuesday, he had 58 - 23 = 35. After losing 2 more on Wednesday, he had 35 - 2 = 33 golf balls.\n"
    "#### 33\n\n"

    "Question: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?\n"
    "Answer: Five bagels at $3 each cost 5 * 3 = 15 dollars. Olivia started with $23, so she has 23 - 15 = 8 dollars left.\n"
    "#### 8\n"
)

print("Model:", MODEL_ID)
print("Method:", METHOD_NAME)
print("Random sampling seed:", SHARED_SEED)
print("GSM8K/ARC/HellaSwag random example target:", QA_EVAL_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)
print("H2O heavy ratio:", HEAVY_RATIO, "| recent ratio:", RECENT_RATIO,
      "| total budget ratio:", HEAVY_RATIO + RECENT_RATIO)

# NOTE: ARC-Challenge/HellaSwag no longer take few-shot exemplars or a
# capped generation length -- they are scored zero-shot via teacher-forced
# per-choice likelihood (see the Shared multiple-choice (MC) scoring
# machinery cell, below), matching large_sample_implementations exactly.
# GSM8K is unaffected: it keeps GSM8K_FEWSHOT_PREFIX above.


In [ ]:
# Block - Load tokenizer + the stock (unmodified) model.
# Identical loading code and kwargs to the KVQuant-family notebooks, with ONE
# method-required difference: attn_implementation="eager". H2O needs the real
# per-step attention weights (output_attentions=True) to score heavy hitters,
# and neither FlashAttention nor SDPA can return them -- eager is a hard
# requirement of the method, not a harness choice. Everything else
# (torch_dtype, low_cpu_mem_usage, device_map, use_fast=False tokenizer,
# pad=eos) matches the other notebooks exactly.
#
# NOTE for cross-method latency analysis: because the KVQuant notebooks run
# sdpa, their timing gap vs. this notebook mixes "eager tax" with "H2O tax".
# Run the baseline notebook once with attn_implementation="eager" to separate
# the two.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_model_kwargs = {
    "torch_dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True,
    "attn_implementation": "eager",   # H2O method requirement (see note above)
    "trust_remote_code": True,
}
if HAS_CUDA:
    _model_kwargs["device_map"] = {"": 0}

print("Loading stock model (H2O modifies the cache at runtime, not the model)...")
model_h2o = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
if not HAS_CUDA:
    model_h2o = model_h2o.to(DEVICE)
model_h2o.eval()
model_h2o.config.use_cache = True

device = next(model_h2o.parameters()).device
print("model_h2o loaded on:", device,
      "| dtype:", next(model_h2o.parameters()).dtype,
      "| attn: eager (required by H2O)")

In [ ]:
# Repo setup - Clone the KVCacheCompression repo (always fresh, matching the
# KVQuant notebooks' clean-clone convention) and initialize ONLY the H2O
# submodule -- the official FMInference/H2O repository. The other submodules
# (KVQuant, KIVI, SnapKV, ...) are not needed here, so a targeted init keeps
# this much faster than --recurse-submodules.
#
# From the submodule we import H2OKVCache_LayerWise from
# h2o_hf/utils_real_drop/modify_llama.py: the authors' real-KV-dropping
# eviction engine (per-layer, per-head heavy-hitter scores; physical pruning
# of the cache tensors). The module's other contents (H2OLlamaAttention /
# H2OLlamaForCausalLM) target an older transformers API and MHA-only models,
# so they are not used -- see the notebook intro for details.

if os.path.exists("/content/KVCacheCompression"):
    shutil.rmtree("/content/KVCacheCompression")
    print("Removed existing repo copy for a clean re-clone")

!git clone https://github.com/yoshikodes/KVCacheCompression.git /content/KVCacheCompression
%cd /content/KVCacheCompression
!git submodule update --init --depth 1 H2O

_h2o_module_path = "/content/KVCacheCompression/H2O/h2o_hf/utils_real_drop/modify_llama.py"
assert os.path.exists(_h2o_module_path), \
    "ERROR: official H2O module not found -- clone or submodule init may have failed."

sys.path.insert(0, "/content/KVCacheCompression/H2O/h2o_hf")
from utils_real_drop.modify_llama import H2OKVCache_LayerWise

print("Imported official H2O engine:", H2OKVCache_LayerWise,
      "\nfrom", _h2o_module_path)

## Helper Functions

Shared inference machinery used across all three datasets (GSM8K,
ARC-Challenge, HellaSwag).

In [ ]:
# Block - sync_if_cuda/clear_memory: used across every timed inference
# loop in this notebook (GSM8K, ARC-Challenge, HellaSwag) for timing-safe
# GPU synchronization and between-dataset memory cleanup.


def sync_if_cuda():
    if HAS_CUDA:
        torch.cuda.synchronize()


def clear_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

In [ ]:
# Block - H2O machinery around the OFFICIAL engine -- shared inference
# machinery used across all three datasets (GSM8K, ARC-Challenge,
# HellaSwag). The eviction algorithm itself lives entirely in the
# submodule's H2OKVCache_LayerWise (imported above, unmodified): per-head
# heavy-hitter score accumulation, per-head top-k + recent keep-set
# selection, physical KV dropping. This cell only provides the plumbing
# that feeds it:
#
#   make_h2o_engines(total_len)
#       One fresh engine per layer (fresh hh_score), exactly as the official
#       H2OLlamaAttention holds one engine per layer. hh_size/recent_size are
#       derived from HEAVY_RATIO/RECENT_RATIO of this sample's total length.
#
#   reduce_attn_to_kv_heads(att)
#       The one GQA adaptation (submodule code untouched): the engine expects
#       one score row per cached KV head, so the 32 query heads' attention
#       weights are summed within each group of 8 sharing a KV head.
#
#   apply_engines(pkv, attentions, engines)
#       Feeds each layer's (legacy KV tuple, reduced attention) through that
#       layer's engine -- the same call H2OLlamaAttention.forward makes
#       (self.kv_cache(past_key_value, attn_weights)) -- and rebuilds the
#       legacy cache.
#
#   h2o_prefill(input_ids, engines)
#       ONE batched forward over the whole prompt (matching the batched
#       prefill inside model.generate() that the KVQuant notebooks use, and
#       matching how the official attention class natively handles
#       multi-token prefill: the full [.., q_len, kv_len] attention map goes
#       into the engine, whose _update_hh_score sums over the query rows),
#       then one engine pass (prune to budget).
#
#   h2o_step(token_tensor, abs_pos, pkv, engines)
#       One decode step: forward one token with explicit position_ids /
#       attention_mask (so RoPE positions stay absolute despite eviction --
#       cached keys were rotated at their original positions before caching,
#       so pruning preserves their correctness), then one engine pass.
#       Handles pkv=None (first step of the teacher-forced loop).
#
# make_h2o_engines is called identically for all three datasets: GSM8K,
# ARC-Challenge, and HellaSwag all size their budget from
# make_h2o_engines(prompt_len), this sample's own prompt length at prefill
# time. There is no GSM8K-specific sizing machinery -- an earlier version
# of this notebook sized GSM8K's budget from a running average of
# previously completed questions' true total length (prompt + generated
# tokens); that history-dependent scheme has been retired in favor of the
# same prompt-length-only sizing every other dataset already used, so
# budgets are now defined uniformly as a fraction of context length, not
# excused as "generation is negligible" (not true for GSM8K's up to
# GSM8K_MAX_NEW_TOKENS=256 generated tokens).

import contextlib
import io

N_LAYERS = model_h2o.config.num_hidden_layers
N_KV_HEADS = int(getattr(model_h2o.config, "num_key_value_heads",
                         model_h2o.config.num_attention_heads))
N_Q_HEADS = model_h2o.config.num_attention_heads
KV_GROUP = N_Q_HEADS // N_KV_HEADS
print(f"Layers: {N_LAYERS} | query heads: {N_Q_HEADS} | KV heads: {N_KV_HEADS} "
      f"| GQA group size: {KV_GROUP}")


def cache_to_legacy(past_key_values):
    if past_key_values is None:
        return None
    if isinstance(past_key_values, tuple):
        return past_key_values
    if hasattr(past_key_values, "to_legacy_cache"):
        return past_key_values.to_legacy_cache()
    raise TypeError(f"Unsupported cache type: {type(past_key_values)}")


def get_cache_tokens(past_key_values):
    past_key_values = cache_to_legacy(past_key_values)
    if past_key_values is None:
        return 0
    return int(past_key_values[0][0].shape[2])


def dense_kv_cache_bytes(past_key_values):
    """Real bytes of the tensors actually resident in past_key_values right
    now, at their current dtype. The official engine physically drops
    positions, so this is an honest MEASURED number (the quantized KVQuant
    notebooks, by contrast, report CALCULATED bytes because their
    compression is simulated)."""
    past_key_values = cache_to_legacy(past_key_values)
    if past_key_values is None:
        return 0
    total = 0
    for key, value in past_key_values:
        total += key.numel() * key.element_size()
        total += value.numel() * value.element_size()
    return int(total)


def kv_bytes_per_token(past_key_values):
    """Exact real bytes one cached token occupies (uniform across positions:
    fp16 K + fp16 V for every layer). Used to convert a tracked max cache
    length into peak bytes without touching the GPU inside timed regions."""
    n_tokens = get_cache_tokens(past_key_values)
    if n_tokens == 0:
        return 0.0
    return dense_kv_cache_bytes(past_key_values) / n_tokens


def make_h2o_engines(total_len):
    """One fresh official engine per layer, budgets from this sample's total
    length. The engine's constructor print ("H2OKVCache-LayerWise: hh, rec")
    is suppressed -- it would fire n_layers times per sample; the budgets are
    reported once by the callers instead."""
    hh_size = max(1, int(total_len * HEAVY_RATIO))
    recent_size = max(1, int(total_len * RECENT_RATIO))
    with contextlib.redirect_stdout(io.StringIO()):
        engines = [
            H2OKVCache_LayerWise(hh_size=hh_size, recent_size=recent_size,
                                 k_seq_dim=2, v_seq_dim=2)
            for _ in range(N_LAYERS)
        ]
    return engines


def reduce_attn_to_kv_heads(att):
    """[batch, n_q_heads, q_len, kv_len] -> [batch, n_kv_heads, q_len, kv_len]
    by summing the query heads within each GQA group: the total attention
    mass received by that KV head's cache entries. (In HF Llama, query head h
    reads KV head h // KV_GROUP, i.e. groups are contiguous blocks.) On MHA
    models KV_GROUP == 1 and this is the identity."""
    if KV_GROUP == 1:
        return att
    b, h, q, kv = att.shape
    return att.view(b, N_KV_HEADS, KV_GROUP, q, kv).sum(dim=2)


def apply_engines(past_key_values, attentions, engines):
    """One official-engine call per layer -- the same call the official
    H2OLlamaAttention.forward makes: self.kv_cache(past_key_value,
    attn_weights). Updates that layer's per-head hh_score and physically
    prunes that layer's KV if over budget."""
    past_key_values = cache_to_legacy(past_key_values)
    new_layers = []
    for layer_kv, layer_att, engine in zip(past_key_values, attentions, engines):
        att_kv = reduce_attn_to_kv_heads(layer_att.detach())
        pruned = engine(tuple(layer_kv), att_kv)
        new_layers.append(tuple(pruned))
    return tuple(new_layers)


@torch.no_grad()
def h2o_prefill(input_ids, engines):
    """Returns (last_logits, pruned_pkv, prefill_cache_tokens).
    prefill_cache_tokens is the cache length BEFORE the post-prefill prune --
    a transient, pre-eviction size that batched-prefill H2O necessarily
    computes along the way. NOT used by this notebook's own
    generate_gsm8k_h2o/generate_mcq_h2o, which inline their own prefill
    instead so the TTFT timestamp can land between the forward and the
    engine pass -- kept here for reference/reuse only. In particular,
    prefill_cache_tokens is NEVER what this notebook reports as
    peak_memory_mb: every peak_memory_mb value in this notebook's results
    is measured strictly AFTER eviction has run, exactly like
    ARC-Challenge/HellaSwag, which never have a pre-eviction moment to
    begin with."""
    outputs = model_h2o(
        input_ids=input_ids,
        use_cache=True,
        output_attentions=True,
        return_dict=True,
    )
    last_logits = outputs.logits[:, -1, :]
    pkv = cache_to_legacy(outputs.past_key_values)
    prefill_cache_tokens = get_cache_tokens(pkv)
    pkv = apply_engines(pkv, outputs.attentions, engines)
    return last_logits, pkv, prefill_cache_tokens


@torch.no_grad()
def h2o_step(token_tensor, abs_pos, past_key_values, engines):
    """One decode step through the H2O-compressed cache."""
    cache_len_before = get_cache_tokens(past_key_values)

    attention_mask = torch.ones(
        (1, cache_len_before + 1),
        dtype=torch.long,
        device=device,
    )
    position_ids = torch.tensor([[abs_pos]], dtype=torch.long, device=device)

    outputs = model_h2o(
        input_ids=token_tensor,
        past_key_values=past_key_values,
        attention_mask=attention_mask,
        position_ids=position_ids,
        use_cache=True,
        output_attentions=True,
        return_dict=True,
    )

    past_key_values = apply_engines(
        outputs.past_key_values, outputs.attentions, engines,
    )
    return outputs.logits[:, -1, :], past_key_values


_demo_total_len = 2048  # representative sequence length, purely for this printout
_demo = make_h2o_engines(_demo_total_len)
print("Official H2O engines ready. Example budgets at total_len =", _demo_total_len, ":",
      f"hh_size {_demo[0].hh_size}, recent_size {_demo[0].recent_size},",
      f"cache_size {_demo[0].cache_size} tokens",
      f"({(_demo[0].cache_size / _demo_total_len):.1%} of dense)")
del _demo

In [ ]:
def seeded_subset(items, max_samples, seed=SHARED_SEED):
    """Select a reproducible random subset, then restore source order.

    A fresh RNG is created on every call, so running notebook sections in a
    different order cannot change which examples are selected.
    """
    items = list(items)

    sample_count = min(int(max_samples), len(items))

    selected_indices = sorted(
        random.Random(int(seed)).sample(range(len(items)), sample_count)
    )

    return [items[index] for index in selected_indices], selected_indices

In [ ]:
def robust_call(fn, *args, max_retries=5, backoff_sec=5, desc="operation", on_retry=None, **kwargs):
    """Retries fn(*args, **kwargs) on any exception, up to max_retries times,
    waiting backoff_sec between attempts -- guards dataset downloads against
    transient network failures (e.g. IncompleteRead/ChunkedEncodingError)
    rather than letting one flaky connection kill the whole notebook run.
    If on_retry is given, it's called (no args) after each failure, before
    the next attempt -- e.g. clear_hf_dataset_cache, so a retry that hit a
    stuck/corrupted partial download actually starts fresh instead of
    resuming (and re-breaking at) the same point every time."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            _msg = f"  {desc}: attempt {attempt}/{max_retries} failed ({e!r})"
            if attempt < max_retries:
                _msg += f", retrying in {backoff_sec}s..."
            print(_msg)
            if attempt < max_retries:
                if on_retry is not None:
                    on_retry()
                time.sleep(backoff_sec)
    raise last_err

In [ ]:
# ============================================================================
# Shared multiple-choice (MC) scoring machinery -- ARC-Challenge and
# HellaSwag are scored via teacher-forced per-choice likelihood, matching the
# large_sample_implementations family's methodology exactly (not
# generate-and-extract): every answer choice is teacher-forced token-by-token
# through the official H2O engine (h2o_step, eviction happening INSIDE the
# timed bracket, exactly like everywhere else in this notebook), and its
# per-token cross-entropy is summed into that choice's NLL. The model is
# never asked to generate anything for these two datasets -- there is no
# few-shot prefix and no chain-of-thought either; both datasets are
# zero-shot. GSM8K is untouched -- it keeps its own separate generative
# grading (generate_gsm8k_h2o, above).
#
# lm_eval_encode_pair -- identical to the large_sample_implementations
# family (same joint-tokenization + splitting logic, matching
# lm-evaluation-harness).
#
# score_mc_choice_h2o(prompt, choice)
#     Times the full step-by-step walk over one choice's tokens via h2o_step.
#     Each choice gets its OWN independently-sized engine budget from its OWN
#     token length (make_h2o_engines(total_len) for THIS choice) -- not a
#     budget shared across a question's choices -- matching how every other
#     dataset in this notebook sizes its H2O budget as a ratio of that
#     specific processed sequence's own true length. Memory is the MEASURED
#     peak cache size, tracked incrementally via cheap shape reads outside
#     the timing fences (same technique as the WikiText-style step loops).
#
# score_mc_question_h2o(prompt, choices, gold_index)
#     Same aggregation as the baseline/KVQuant notebooks: raw/normalized
#     accuracy, perplexity from the gold choice only, TTFT = mean across
#     choices, TBT = weighted mean across every choice's decode steps, total
#     latency = SUM across choices, peak memory = MAX across choices.
# ============================================================================


def lm_eval_encode_pair(context, choice):
    context = str(context)
    continuation = " " + str(choice)

    n_spaces = len(context) - len(context.rstrip())
    if n_spaces > 0:
        continuation = context[-n_spaces:] + continuation
        context = context[:-n_spaces]

    if not context:
        raise ValueError("MC context cannot be empty.")

    whole_ids = tokenizer(context + continuation, add_special_tokens=True)["input_ids"]
    context_ids = tokenizer(context, add_special_tokens=True)["input_ids"]
    continuation_ids = whole_ids[len(context_ids):]

    if not context_ids:
        raise ValueError("Context tokenization produced no tokens.")
    if not continuation_ids:
        raise ValueError(f"Continuation tokenization produced no tokens. Context={context!r}, choice={choice!r}")

    return context_ids, continuation_ids

@torch.no_grad()
def score_mc_choice_h2o(prompt, choice):
    context_ids, continuation_ids = lm_eval_encode_pair(prompt, choice)
    full_ids_1d = torch.tensor(context_ids + continuation_ids, device=device)
    n_context = len(context_ids)

    engines = make_h2o_engines(full_ids_1d.shape[0])

    chunk_ids = full_ids_1d.unsqueeze(0)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    scored = 0
    step_times = []
    past_key_values = None
    max_cache_tokens = 0

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        step_logits, past_key_values = h2o_step(step_input, pos, past_key_values, engines)
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        step_target = target_ids[:, pos]

        if pos + 1 >= n_context:
            loss = loss_fct(step_logits, step_target)
            nll_sum += loss.float().item()
            scored += 1

        max_cache_tokens = max(max_cache_tokens, get_cache_tokens(past_key_values))

    ttft_sec = step_times[0]
    decode_time_sum = sum(step_times[1:])
    decode_steps = len(step_times) - 1
    total_latency_sec = sum(step_times)

    peak_bytes = int(round(max_cache_tokens * kv_bytes_per_token(past_key_values)))

    return {
        "nll_sum": nll_sum, "scored": scored,
        "ttft_sec": ttft_sec, "decode_time_sum": decode_time_sum, "decode_steps": decode_steps,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
        "choice_char_len": max(len(str(choice)), 1),
    }


@torch.no_grad()
def score_mc_question_h2o(prompt, choices, gold_index):
    choice_results = [score_mc_choice_h2o(prompt, choice) for choice in choices]

    normalized_nlls = [r["nll_sum"] / r["choice_char_len"] for r in choice_results]

    raw_prediction = int(min(range(len(choice_results)), key=lambda i: choice_results[i]["nll_sum"]))
    normalized_prediction = int(min(range(len(choice_results)), key=lambda i: normalized_nlls[i]))

    gold_result = choice_results[gold_index]
    gold_mean_nll = gold_result["nll_sum"] / max(gold_result["scored"], 1)

    total_decode_time = sum(r["decode_time_sum"] for r in choice_results)
    total_decode_steps = sum(r["decode_steps"] for r in choice_results)

    return {
        "raw_prediction": raw_prediction,
        "normalized_prediction": normalized_prediction,
        "raw_correct": int(raw_prediction == gold_index),
        "normalized_correct": int(normalized_prediction == gold_index),
        "perplexity": math.exp(min(gold_mean_nll, 50.0)),
        "ttft_sec": sum(r["ttft_sec"] for r in choice_results) / len(choice_results),
        "tbt_sec": (total_decode_time / total_decode_steps) if total_decode_steps > 0 else 0.0,
        "total_latency_sec": sum(r["total_latency_sec"] for r in choice_results),
        "peak_memory_bytes": max(r["peak_memory_bytes"] for r in choice_results),
    }


## GSM8K

In [ ]:
# Block - GSM8K loading: combine all splits, then random sampling.
# Load train + test splits, build the complete list of valid question/answer
# pairs, then select up to 2,048 examples with seed 42. This creates a
# reproducible random sample across the entire GSM8K dataset.

def extract_gsm8k_gold_answer(answer_text):
    match = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", answer_text)
    if not match:
        return None
    try:
        return float(match.group(1).replace(",", ""))
    except ValueError:
        return None

gsm8k_train = robust_call(
    load_dataset,
    "gsm8k",
    "main",
    split="train",
    desc="GSM8K train load",
    on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

gsm8k_test = robust_call(
    load_dataset,
    "gsm8k",
    "main",
    split="test",
    desc="GSM8K test load",
    on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

# Combine all official GSM8K splits
gsm8k_all = list(gsm8k_train) + list(gsm8k_test)
all_gsm8k_pairs = []

for item in gsm8k_all:
    gold = extract_gsm8k_gold_answer(item["answer"])
    if gold is not None:
        all_gsm8k_pairs.append({
            "question": item["question"],
            "gold": gold,
            "gold_text": item["answer"],
        })

gsm8k_qa_pairs, gsm8k_selected_indices = seeded_subset(
    all_gsm8k_pairs,
    QA_EVAL_SAMPLES,
    SHARED_SEED,
)

print(
    f"GSM8K: {len(all_gsm8k_pairs)} valid questions available; "
    f"selected {len(gsm8k_qa_pairs)} random questions "
    f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
)

print(
    "GSM8K selected valid-item indices (first 20):",
    gsm8k_selected_indices[:20]
)

In [ ]:
# Block - GSM8K generation with H2O (official engines). Prefill processes the
# whole prompt in ONE batched forward (exactly like the prefill inside
# model.generate() that the KVQuant notebooks use, and exactly how the
# official H2OLlamaAttention handles multi-token inputs: the full
# [.., q_len, kv_len] attention map goes into the engine, whose
# _update_hh_score sums over the query rows); decode then continues greedily
# one token at a time with per-step eviction.
#
# Budget sizing: the engine budget is sized from THIS QUESTION'S OWN prompt
# length alone -- make_h2o_engines(prompt_len), identical to how
# ARC-Challenge/HellaSwag size their budget (generate_mcq_h2o) and to the
# H2O machinery cell's own docstring. There is no lookahead into how many
# tokens will be generated, and no dependency on other questions' history
# (an earlier version of this notebook sized GSM8K's budget from a running
# average of previously completed questions' true total length -- that has
# been retired). Concretely: GSM8K's budget is a fraction of PROMPT length
# only, not of prompt+generation length, so long generations (up to
# GSM8K_MAX_NEW_TOKENS=256 tokens) decode against a cache budget that was
# fixed before any of those tokens existed. That is a real property of this
# budgeting policy worth keeping in mind when reading GSM8K's numbers -- it
# is not excused here as "negligible", since 256 tokens is not a negligible
# fraction of a typical GSM8K few-shot prompt.
#
# Timing matches the KVQuant notebooks' definitions exactly:
#   TTFT  = time from generation start until the first generated token's
#           logits are ready (i.e., the batched prefill forward) -- the same
#           quantity their StoppingCriteria timestamp captures.
#   total = wall time of the whole generation.
#   TBT   = (total - TTFT) / max(n_generated - 1, 1) -- the same formula.
# The post-prefill engine pass lands AFTER the TTFT timestamp (the first
# token comes straight from the prefill logits and does not need the prune),
# so it is counted in total latency / TBT as decode-phase method overhead.
#
# n_generated counts every token an actual timed forward call produced
# (prefill's token + one per h2o_step call), INCLUDING a final EOS token
# that ends the loop -- matching the KVQuant notebooks' StoppingCriteria,
# which fires after the EOS-producing step completes too, before
# generate() checks for termination. generated_ids/gen_text deliberately
# excludes EOS itself (matching skip_special_tokens=True decoding
# elsewhere), but n_generated must still count that step's forward call,
# or tbt_sec's denominator undercounts by one relative to KVQuant every
# time a question ends via EOS rather than hitting GSM8K_MAX_NEW_TOKENS --
# tracked separately below as n_forward_tokens.
#
# Answer grading is VERBATIM from the KVQuant notebooks: truncate the
# generation at the first "Question:", then extract "#### <num>" (falling
# back to the last number), compare |pred - gold| < 1e-4.
#
# Memory: MEASURED peak = max cache length observed, tracked via cheap
# shape reads taken ONLY after apply_engines has pruned that step's cache
# -- never the transient, budget-independent moment that exists right
# after the batched prefill forward but before eviction has run (a single
# forward over the whole prompt necessarily computes a full, unpruned
# cache for every prompt token, since H2O needs the resulting attention
# scores to decide what to evict in the first place -- that computation is
# unavoidable, but nothing downstream ever stores or reuses that
# pre-eviction snapshot: pkv is replaced by the pruned version immediately
# after). This matches exactly how ARC-Challenge/HellaSwag already measure
# peak (they never have a pre-eviction moment to begin with, since they
# evict from the very first token), so peak/average memory are directly
# comparable, and budget-sensitive, across all three datasets.
#
# Perplexity is measured on the model's OWN generated answer, live from
# this same hand-rolled decode loop -- no separate teacher-forced pass.
# Per-step logits (last_logits) are already computed for greedy decoding
# regardless, so detecting the answer span costs one extra cheap text
# decode per token (inline, can't be deferred in a hand-rolled loop); the
# actual log-probability scoring is deferred until after gen_end below,
# using logits saved during the loop -- no extra forward pass, and no
# additional cost inside the timed window beyond that cheap text check.
# The span starts right after four consecutive '#' characters (the
# "####" marker GSM8K answers use) and stops the moment another '#'
# appears. If "####" never appears, perplexity is None for that question.


def _extract_final_number(text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", text)
    if m:
        num_str = m.group(1)
    else:
        nums = re.findall(r"-?[0-9][0-9,]*\.?[0-9]*", text)
        if not nums:
            return None
        num_str = nums[-1]
    num_str = num_str.replace(",", "").rstrip(".")
    try:
        return float(num_str)
    except ValueError:
        return None


@torch.no_grad()
def generate_gsm8k_h2o(question):
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    engines = make_h2o_engines(prompt_len)

    sync_if_cuda()
    gen_start = time.perf_counter()

    # ---- Prefill (one batched forward; first token's logits ready here).
    # Done inline rather than via h2o_prefill so the TTFT timestamp can land
    # between the forward and the engine pass -- see header comment. ----
    outputs = model_h2o(input_ids=enc["input_ids"], use_cache=True,
                        output_attentions=True, return_dict=True)
    last_logits = outputs.logits[:, -1, :]
    next_id = int(last_logits.argmax(dim=-1)[0].item())
    sync_if_cuda()
    ttft_sec = time.perf_counter() - gen_start

    # ---- Official-engine pass on the prefill attention map, then prune
    # (counted in total latency, not TTFT -- see header comment) ----
    pkv = cache_to_legacy(outputs.past_key_values)
    pkv = apply_engines(pkv, outputs.attentions, engines)

    max_cache_tokens = get_cache_tokens(pkv)  # already post-eviction -- see header comment
    generated_ids = []
    n_forward_tokens = 1  # the prefill forward already produced next_id -- see header comment

    # ---- Live perplexity-span detection state -- see header comment ----
    hash_streak = 0
    in_answer_span = False
    answer_done = False
    answer_token_ids = []
    answer_logits = []

    # ---- Greedy decode with per-step eviction ----
    for step in range(GSM8K_MAX_NEW_TOKENS):
        if next_id == tokenizer.eos_token_id:
            break
        generated_ids.append(next_id)

        if not answer_done:
            tok_text = tokenizer.decode([next_id])
            if in_answer_span:
                if "#" in tok_text:
                    answer_done = True
                else:
                    answer_token_ids.append(next_id)
                    answer_logits.append(last_logits.detach())
            else:
                for ch in tok_text:
                    hash_streak = hash_streak + 1 if ch == "#" else 0
                if hash_streak >= 4:
                    in_answer_span = True
        if "\n" in tokenizer.decode([next_id]):        # early stop: halt after the #### answer line
            _run_txt = tokenizer.decode(generated_ids)
            if "####" in _run_txt and "\n" in _run_txt.split("####", 1)[-1]:
                break

        if step == GSM8K_MAX_NEW_TOKENS - 1:
            break  # no wasted forward for a token we would never use

        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        abs_pos = prompt_len + step
        last_logits, pkv = h2o_step(token_tensor, abs_pos, pkv, engines)
        next_id = int(last_logits.argmax(dim=-1)[0].item())
        n_forward_tokens += 1

        cache_tokens = get_cache_tokens(pkv)  # shape read only, already post-eviction
        max_cache_tokens = max(max_cache_tokens, cache_tokens)

    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    # Perplexity of the model's own predicted final number, scored here
    # (fully after gen_end, untimed) from the logits saved above.
    if answer_token_ids:
        nll_sum = 0.0
        for tok_id, logits in zip(answer_token_ids, answer_logits):
            log_probs = torch.log_softmax(logits[0].float(), dim=-1)
            nll_sum += -log_probs[tok_id].item()
        perplexity = math.exp(min(nll_sum / len(answer_token_ids), 50.0))
    else:
        perplexity = None

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    gen_text = gen_text.split("Question:")[0]

    n_generated = n_forward_tokens
    if len(generated_ids) == 0:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated

    bpt = kv_bytes_per_token(pkv)
    peak_bytes = int(round(max_cache_tokens * bpt))

    return {
        "full_prompt": prompt, "prefill_tokens": prompt_len, "generated_tokens": len(generated_ids), "gen_text": gen_text, "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "total_tokens": total_tokens,
        "peak_memory_bytes": peak_bytes,
        "perplexity": perplexity,
    }


In [ ]:
# Block - GSM8K driver: run every question through generate_gsm8k_h2o
# (accuracy + TTFT/TBT/latency + memory + perplexity, all from the SAME
# real generation call -- perplexity is measured live on the model's own
# generated answer, no separate teacher-forced pass). Aggregation is
# identical to the KVQuant notebooks: accuracy = fraction correct,
# perplexity = MEAN of per-question perplexities, TTFT/TBT/latency = means,
# peak_memory_mb = max over questions, average_memory_mb = mean over
# questions. Same output columns.


def evaluate_gsm8k_h2o(qa_pairs, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_question_records = []

    N_PREVIEW_QUESTIONS = 1000
    for q_idx, qa in enumerate(tqdm(qa_pairs, desc=f"GSM8K | {method_label}")):
        result = generate_gsm8k_h2o(qa["question"])
        pred = _extract_final_number(result["gen_text"])
        is_correct = pred is not None and abs(pred - qa["gold"]) < 1e-4
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- GSM8K | {method_label} | question {q_idx} preview ---")
            print(f"Question:    {qa['question']}")
            print(f"Generated:   {result['gen_text'].strip()}")
            print(f"Gold answer: {qa['gold']} | Predicted: {pred} | Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

        ppl = result["perplexity"]
        if ppl is not None:
            ppl_values.append(ppl)

        per_question_records.append({
            "question_index": q_idx,
            "full_prompt": result["full_prompt"],
            "prefill_tokens": result["prefill_tokens"],
            "generated_output": result["gen_text"],
            "generated_tokens": result["generated_tokens"],
            "gold_answer": qa["gold"],
            "predicted_answer": pred,
            "correct": int(is_correct),
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_gsm8k_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_question_records)} per-question GSM8K rows to {_per_prompt_path}")

    return {
        "dataset": "GSM8K",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


gsm8k_results = [
    evaluate_gsm8k_h2o(gsm8k_qa_pairs, METHOD_NAME),
]
gsm8k_results_df = pd.DataFrame(gsm8k_results)
display(gsm8k_results_df)


## ARC-Challenge

In [ ]:
# Block - ARC-Challenge loading: ALL official splits (train + validation +
# test) combined, then random sampling -- mirrors GSM8K's own loading exactly
# (GSM8K combines train+test since it has no validation split; ARC-Challenge
# has all three, so all three go in). Every row across the combined pool gets
# the same validity filtering, then seeded_subset draws up to QA_EVAL_SAMPLES
# reproducibly. Scored by teacher-forced per-choice likelihood via
# score_mc_question_* (Helper Functions, above), zero-shot, not generation.


def load_arc_challenge_items():
    arc_train = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="train",
        desc="ARC-Challenge train load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )
    arc_validation = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="validation",
        desc="ARC-Challenge validation load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )
    arc_test = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="test",
        desc="ARC-Challenge test load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )
    arc_all = list(arc_train) + list(arc_validation) + list(arc_test)

    valid_items = []
    for row in arc_all:
        labels = row["choices"]["label"]
        texts = row["choices"]["text"]
        answer_key = row["answerKey"]
        if answer_key not in labels:
            continue
        valid_items.append({
            "question": row["question"],
            "choices": list(zip(labels, texts)),
            "gold_label": answer_key,
        })

    selected_items, selected_indices = seeded_subset(
        valid_items,
        QA_EVAL_SAMPLES,
        SHARED_SEED,
    )
    print(
        f"ARC-Challenge: {len(valid_items)} valid questions available "
        f"(train+validation+test combined); selected {len(selected_items)} "
        f"random questions (requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("ARC-Challenge selected valid-item indices (first 20):", selected_indices[:20])
    return selected_items


arc_items = load_arc_challenge_items()


In [ ]:
# Block - ARC-Challenge driver: scores every answer choice for each
# question via the shared score_mc_question_h2o (Helper Functions),
# then reports character-length normalized MC accuracy -- matches
# large_sample_implementations' evaluate_arc_kvquant/h2o methodology exactly
# (teacher-forced per-choice likelihood, not generation). Aggregation mirrors
# GSM8K: perplexity = MEAN of per-question perplexities (from each question's
# gold choice), TTFT/TBT/latency = means over questions, peak_memory_mb = max
# over questions, average_memory_mb = mean over questions.


def evaluate_arc_h2o(items, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_item_records = []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"ARC-Challenge | {method_label}")):
        prompt = f"Question: {item['question']}\nAnswer:"
        choice_texts = [text for _, text in item["choices"]]
        gold_index = next(i for i, (label, _) in enumerate(item["choices"]) if label == item["gold_label"])

        result = score_mc_question_h2o(prompt, choice_texts, gold_index)

        correct += result["normalized_correct"]
        total += 1

        predicted_label = item["choices"][result["normalized_prediction"]][0]
        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- ARC-Challenge | {method_label} | item {idx} preview ---")
            print(f"Question:   {item['question']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold label: {item['gold_label']} | Predicted: {predicted_label} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

        choices_block = "\n".join(f"{label}. {text}" for label, text in item["choices"])
        prompt_and_choices = f"{item['question']}\n{choices_block}"

        per_item_records.append({
            "item_index": idx,
            "prompt": prompt_and_choices,
            "gold_label": item["gold_label"],
            "predicted_label": predicted_label,
            "correct": result["normalized_correct"],
            "correct_raw": result["raw_correct"],
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_arc_challenge_per_prompt.csv"
    pd.DataFrame(per_item_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_item_records)} per-item ARC-Challenge rows to {_per_prompt_path}")

    return {
        "dataset": "ARC-Challenge",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


arc_results = [
    evaluate_arc_h2o(arc_items, METHOD_NAME),
]
arc_results_df = pd.DataFrame(arc_results)
display(arc_results_df)


## HellaSwag

In [ ]:
# Block - HellaSwag loading: ALL LABELED official splits (train +
# validation) combined, then random sampling -- mirrors GSM8K's own loading
# as closely as HellaSwag allows: HellaSwag's test split ships unlabeled
# (label == -1, no gold answer to score against), so it cannot be included
# the way GSM8K's test split is; train + validation is the full labeled pool
# available. Every row across the combined pool gets the same validity
# filtering, then seeded_subset draws up to QA_EVAL_SAMPLES reproducibly.
# Scored by teacher-forced per-choice likelihood via score_mc_question_*
# (Helper Functions, above), zero-shot, not generation.


def hellaswag_preprocess(text):
    text = str(text).strip()
    text = text.replace(" [title]", ". ")
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace("  ", " ")
    return text


def _hs_valid(item):
    label = str(item.get("label", "")).strip()
    endings = item.get("endings", [])
    return label.isdigit() and len(endings) >= 2 and 0 <= int(label) < len(endings)


def load_hellaswag_items():
    hs_train = robust_call(
        load_dataset, "Rowan/hellaswag", split="train",
        desc="HellaSwag train load", on_retry=lambda: clear_hf_dataset_cache("hellaswag"),
    )
    hs_validation = robust_call(
        load_dataset, "Rowan/hellaswag", split="validation",
        desc="HellaSwag validation load", on_retry=lambda: clear_hf_dataset_cache("hellaswag"),
    )
    hs_all = list(hs_train) + list(hs_validation)

    processed_items = []
    for row in hs_all:
        if not _hs_valid(row):
            continue
        context = str(row["ctx_a"]) + " " + str(row["ctx_b"]).capitalize()
        prompt = hellaswag_preprocess(str(row["activity_label"]) + ": " + context)
        choices = [hellaswag_preprocess(choice) for choice in row["endings"]]
        processed_items.append({
            "prompt": prompt,
            "choices": choices,
            "gold_index": int(row["label"]),
        })

    selected_items, selected_indices = seeded_subset(
        processed_items,
        QA_EVAL_SAMPLES,
        SHARED_SEED,
    )
    print(
        f"HellaSwag: {len(processed_items)} valid examples available "
        f"(train+validation combined; test excluded -- unlabeled); "
        f"selected {len(selected_items)} random examples "
        f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("HellaSwag selected example indices (first 20):", selected_indices[:20])
    return selected_items


hellaswag_items = load_hellaswag_items()


In [ ]:
# Block - HellaSwag driver: scores every answer choice for each example
# via the shared score_mc_question_h2o (Helper Functions) -- the
# same machinery ARC-Challenge uses, since both are likelihood-scored MC
# datasets. Aggregation is identical to ARC-Challenge: normalized accuracy,
# perplexity = MEAN of per-question perplexities (from each example's gold
# ending), TTFT/TBT/latency = means over examples, peak_memory_mb = max over
# examples, average_memory_mb = mean over examples.


def evaluate_hellaswag_h2o(items, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_item_records = []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"HellaSwag | {method_label}")):
        result = score_mc_question_h2o(item["prompt"], item["choices"], item["gold_index"])

        correct += result["normalized_correct"]
        total += 1

        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- HellaSwag | {method_label} | item {idx} preview ---")
            print(f"Prompt:     {item['prompt']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold index: {item['gold_index']} | Predicted: {result['normalized_prediction']} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

        letters = "ABCDEFGHIJ"[:len(item["choices"])]
        choices_block = "\n".join(f"{letter}. {choice}" for letter, choice in zip(letters, item["choices"]))
        prompt_and_choices = f"{item['prompt']}\n{choices_block}"

        per_item_records.append({
            "item_index": idx,
            "prompt": prompt_and_choices,
            "gold_index": item["gold_index"],
            "predicted_index": result["normalized_prediction"],
            "correct": result["normalized_correct"],
            "correct_raw": result["raw_correct"],
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_hellaswag_per_prompt.csv"
    pd.DataFrame(per_item_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_item_records)} per-item HellaSwag rows to {_per_prompt_path}")

    return {
        "dataset": "HellaSwag",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


hellaswag_results = [
    evaluate_hellaswag_h2o(hellaswag_items, METHOD_NAME),
]
hellaswag_results_df = pd.DataFrame(hellaswag_results)
display(hellaswag_results_df)


## RULER


In [ ]:
# ============================================================================
# RULER settings + single-needle-in-a-haystack (NIAH) example generation.
#
# RULER (Hsieh et al., "What's the Real Context Size of Your Long-Context
# Language Models?", arXiv:2404.06654) is a SYNTHETIC long-context benchmark
# GENERATOR, not a fixed dataset -- NVIDIA's own reference implementation
# builds examples per-tokenizer/per-length/per-seed rather than shipping one
# canonical pre-built dataset (there is no single official HF dataset
# covering 4K/8K/16K/32K for an arbitrary model). This notebook generates
# RULER's flagship task -- single-needle retrieval (niah_single) -- directly,
# using THIS model's own tokenizer for exact length control, following
# NVIDIA/RULER's own documented recipe (scripts/data/synthetic/niah.py /
# constants.py): a "noise" haystack (RULER's own filler-sentence haystack
# type, not an invented approximation) with one key/value "needle" sentence
# inserted at a random depth, and RULER's own official niah prompt template
# + answer_prefix (verbatim, type_needle_v="numbers").
#
# Length buckets: 4096 / 8192 / 16384 tokens (~683/683/682 samples, 2048
# total). 32768 is DELIBERATELY EXCLUDED: eager attention (required
# throughout this notebook family) materializes the full [heads, seq, seq]
# attention-score matrix, which at 32K tokens is on the order of 60-140GB
# for a SINGLE layer -- a risk shared identically by every compression
# method here (compression shrinks the KV CACHE, not the attention-score
# computation itself, so no method here is protected from it). 16384 is
# still non-trivial (~17GB/layer at bf16) -- if it OOMs on your GPU, shrink
# RULER_LENGTH_BUCKETS below; that is a hardware ceiling, not a code bug.
#
# Grading matches RULER's own convention: does the gold value string appear
# in the model's generated text (substring/recall match), not exact-string
# equality of the whole output.

import uuid

RULER_LENGTH_BUCKETS = [4096, 8192, 16384]
RULER_MAX_NEW_TOKENS = 128  # matches RULER's own niah task config (tokens_to_generate=128)

_ruler_base, _ruler_rem = divmod(QA_EVAL_SAMPLES, len(RULER_LENGTH_BUCKETS))
RULER_SAMPLES_PER_BUCKET = {
    length: _ruler_base + (1 if i < _ruler_rem else 0)
    for i, length in enumerate(RULER_LENGTH_BUCKETS)
}

# RULER's own "noise" haystack type (NVIDIA/RULER niah.py, haystack_type=
# "noise"): a fixed pool of filler sentences, repeated to reach the target
# length.
RULER_HAYSTACK_SENTENCES = [
    "The grass is green.",
    "The sky is blue.",
    "The sun is yellow.",
    "Here we go.",
    "There and back again.",
]
RULER_SENTENCE_TOKENS = {
    s: len(tokenizer(s, add_special_tokens=False)["input_ids"])
    for s in RULER_HAYSTACK_SENTENCES
}

# RULER's own official niah prompt template + answer_prefix (verbatim, from
# NVIDIA/RULER scripts/data/synthetic/constants.py's 'niah' task, with
# type_needle_v="numbers"), plus one explicit generation instruction so the
# model is told exactly how to format its answer -- matching this notebook
# family's convention of stating the expected output format explicitly
# (the same way GSM8K's few-shot prefix states "end with exactly this
# format: #### <final number>").
_RULER_HEADER = (
    "Some special magic numbers are hidden within the following text. Make "
    "sure to memorize them. I will quiz you about the numbers afterwards. "
    "Respond with only the magic number and nothing else.\n"
)
_RULER_QUERY_TAIL = (
    "\nWhat is the special magic number for {key} mentioned in the provided text?"
    "\nAnswer: The special magic number for {key} mentioned in the provided text is"
)


def _ruler_make_needle(rng):
    """RULER's own 'uuids' key type / 'numbers' value type (both official
    RULER type_needle options) -- no external word-list dependency needed."""
    key = str(uuid.UUID(int=rng.getrandbits(128)))
    value = str(rng.randint(1000000, 9999999))  # 7-digit number, RULER's default
    return key, value


def _ruler_build_item(rng, target_tokens):
    key, value = _ruler_make_needle(rng)
    needle_sentence = f"One of the special magic numbers for {key} is: {value}."

    # Build haystack sentences until the filler alone covers target_tokens
    # (a small, roughly-fixed header/query/answer-prefix/needle overhead
    # sits on top -- RULER's own lengths are nominal/approximate too, not
    # exact byte-for-byte token counts).
    sentences = []
    token_count = 0
    while token_count < target_tokens:
        sentence = rng.choice(RULER_HAYSTACK_SENTENCES)
        sentences.append(sentence)
        token_count += RULER_SENTENCE_TOKENS[sentence]

    # Insert the needle at a random depth (0-100% of the haystack), matching
    # RULER's own DEPTHS sampling (NVIDIA/RULER niah.py).
    insert_at = rng.randint(0, len(sentences))
    sentences.insert(insert_at, needle_sentence)
    context = " ".join(sentences)

    prompt = _RULER_HEADER + context + _RULER_QUERY_TAIL.format(key=key)

    return {"prompt": prompt, "key": key, "gold_value": value, "target_tokens": target_tokens}


def build_ruler_items():
    rng = random.Random(SHARED_SEED)
    items = []
    for target_tokens in RULER_LENGTH_BUCKETS:
        n = RULER_SAMPLES_PER_BUCKET[target_tokens]
        for _ in range(n):
            items.append(_ruler_build_item(rng, target_tokens))
    return items


ruler_items = build_ruler_items()
print(
    f"RULER: generated {len(ruler_items)} single-needle items -- "
    + ", ".join(f"{RULER_SAMPLES_PER_BUCKET[l]} @ {l} tokens" for l in RULER_LENGTH_BUCKETS)
)
_ruler_preview = ruler_items[0]
print(f"\nExample prompt (first {RULER_LENGTH_BUCKETS[0]}-token item), truncated:")
print(_ruler_preview["prompt"][:400] + " ...[haystack continues]... " + _ruler_preview["prompt"][-300:])
print(f"\nGold value: {_ruler_preview['gold_value']}")


In [ ]:
# Block - RULER driver: hand-rolled prefill + greedy-decode loop through the
# official H2O engine, identical structure to generate_gsm8k_h2o/
# evaluate_gsm8k_h2o (make_h2o_engines/h2o_step, eviction happening INSIDE
# the timed bracket, same TTFT/TBT/latency definitions), scoring a single
# generated answer per item rather than choices. The engine budget is sized
# from THIS ITEM'S OWN prompt length, exactly like GSM8K/ARC/HellaSwag do.
# Perplexity is the mean NLL of the model's own generated tokens (no
# "####"-style span to isolate here, so it covers the whole short answer
# span, unlike GSM8K's marker-scoped perplexity). Memory is the MEASURED
# peak cache size, tracked live during decode -- no extra pass needed.
#
# CSV note: full_prompt is NOT written per row here (unlike GSM8K) -- a
# RULER prompt is up to ~16K tokens of synthetic haystack text, and writing
# that out 2048 times would bloat the CSV for no analytical benefit;
# length_bucket identifies which of the three context lengths each row is.


@torch.no_grad()
def generate_ruler_h2o(prompt):
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    engines = make_h2o_engines(prompt_len)

    sync_if_cuda()
    gen_start = time.perf_counter()

    outputs = model_h2o(input_ids=enc["input_ids"], use_cache=True,
                        output_attentions=True, return_dict=True)
    last_logits = outputs.logits[:, -1, :]
    next_id = int(last_logits.argmax(dim=-1)[0].item())
    sync_if_cuda()
    ttft_sec = time.perf_counter() - gen_start

    pkv = cache_to_legacy(outputs.past_key_values)
    pkv = apply_engines(pkv, outputs.attentions, engines)

    max_cache_tokens = get_cache_tokens(pkv)
    generated_ids = []
    generated_logits = []
    n_forward_tokens = 1

    for step in range(RULER_MAX_NEW_TOKENS):
        if next_id == tokenizer.eos_token_id:
            break
        generated_ids.append(next_id)
        generated_logits.append(last_logits.detach())
        if "\n" in tokenizer.decode([next_id]):
            break

        if step == RULER_MAX_NEW_TOKENS - 1:
            break

        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        abs_pos = prompt_len + step
        last_logits, pkv = h2o_step(token_tensor, abs_pos, pkv, engines)
        next_id = int(last_logits.argmax(dim=-1)[0].item())
        n_forward_tokens += 1

        cache_tokens = get_cache_tokens(pkv)
        max_cache_tokens = max(max_cache_tokens, cache_tokens)

    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    if generated_ids:
        nll_sum = 0.0
        for tok_id, logits in zip(generated_ids, generated_logits):
            log_probs = torch.log_softmax(logits[0].float(), dim=-1)
            nll_sum += -log_probs[tok_id].item()
        perplexity = math.exp(min(nll_sum / len(generated_ids), 50.0))
    else:
        perplexity = None

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    n_generated = n_forward_tokens
    if len(generated_ids) == 0:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated
    bpt = kv_bytes_per_token(pkv)
    peak_bytes = int(round(max_cache_tokens * bpt))

    return {
        "prefill_tokens": prompt_len, "generated_tokens": len(generated_ids), "gen_text": gen_text,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec, "total_latency_sec": total_latency_sec,
        "total_tokens": total_tokens, "peak_memory_bytes": peak_bytes, "perplexity": perplexity,
    }


def evaluate_ruler_h2o(items, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_question_records = []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, item in enumerate(tqdm(items, desc=f"RULER | {method_label}")):
        result = generate_ruler_h2o(item["prompt"])
        is_correct = item["gold_value"] in result["gen_text"]
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- RULER | {method_label} | question {q_idx} ({item['target_tokens']} tokens) preview ---")
            print(f"Gold value: {item['gold_value']}")
            print(f"Generated:  {result['gen_text'].strip()!r}")
            print(f"Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        if result["perplexity"] is not None:
            ppl_values.append(result["perplexity"])

        per_question_records.append({
            "question_index": q_idx,
            "length_bucket": item["target_tokens"],
            "full_prompt": item["prompt"],
            "prefill_tokens": result["prefill_tokens"],
            "generated_output": result["gen_text"],
            "generated_tokens": result["generated_tokens"],
            "gold_answer": item["gold_value"],
            "correct": int(is_correct),
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_ruler_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_question_records)} per-question RULER rows to {_per_prompt_path}")

    return {
        "dataset": "RULER",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


ruler_results = [
    evaluate_ruler_h2o(ruler_items, METHOD_NAME),
]
ruler_results_df = pd.DataFrame(ruler_results)
display(ruler_results_df)


## SQuAD1.1


In [ ]:
# ============================================================================
# SQuAD1.1 (rajpurkar/squad -- the standard v1.1 release, every question has
# an answer in its passage, unlike squad_v2's unanswerable questions):
# extractive reading comprehension. Loading combines train + validation
# (SQuAD ships no public-label test split), matching the GSM8K/ARC/HellaSwag
# convention of pooling every official labeled split before a seeded sample.
# Zero-shot, generation-based, graded with the OFFICIAL SQuAD metrics --
# Exact Match and F1 -- against the best of however many gold reference
# answers a question has (validation questions often carry more than one
# accepted answer; train questions usually carry exactly one).
# ============================================================================

import re
import string
from collections import Counter

SQUAD_MAX_NEW_TOKENS = 48  # SQuAD gold answers are short spans (usually 1-5 words)

SQUAD_HEADER = (
    "Answer the question based on the passage below. Respond with a short "
    "phrase copied directly from the passage that answers the question, and "
    "nothing else.\n\n"
)


def format_squad_prompt(item):
    return (
        SQUAD_HEADER
        + f"Passage: {item['context'].strip()}\n"
        + f"Question: {item['question'].strip()}\n"
        + "Answer:"
    )


def load_squad_items():
    squad_train = robust_call(
        load_dataset, "rajpurkar/squad", split="train",
        desc="SQuAD1.1 train load", on_retry=lambda: clear_hf_dataset_cache("squad"),
    )
    squad_validation = robust_call(
        load_dataset, "rajpurkar/squad", split="validation",
        desc="SQuAD1.1 validation load", on_retry=lambda: clear_hf_dataset_cache("squad"),
    )
    squad_all = list(squad_train) + list(squad_validation)

    valid_items = []
    for row in squad_all:
        texts = row.get("answers", {}).get("text", [])
        gold_answers = [str(t).strip() for t in texts if str(t).strip()]
        if not gold_answers or not str(row.get("context", "")).strip() or not str(row.get("question", "")).strip():
            continue
        valid_items.append({
            "context": row["context"],
            "question": row["question"],
            "gold_answers": gold_answers,
        })

    selected_items, selected_indices = seeded_subset(valid_items, QA_EVAL_SAMPLES, SHARED_SEED)
    print(
        f"SQuAD1.1: {len(valid_items)} valid questions available "
        f"(train+validation combined); selected {len(selected_items)} "
        f"random questions (requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("SQuAD1.1 selected valid-item indices (first 20):", selected_indices[:20])
    return selected_items


squad_items = load_squad_items()


# ---------------------------------------------------------------------------
# Official SQuAD1.1 scoring (Rajpurkar et al. 2016's own normalize_answer /
# exact_match_score / f1_score, reproduced verbatim): lowercase, drop
# punctuation, drop articles (a/an/the), collapse whitespace, then compare.
# Both metrics take the MAX over every gold reference answer available for
# that question.
# ---------------------------------------------------------------------------

def _squad_normalize(text):
    text = text.lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = " ".join(text.split())
    return text


def squad_exact_match(prediction, gold_answers):
    norm_pred = _squad_normalize(prediction)
    return max(int(norm_pred == _squad_normalize(g)) for g in gold_answers)


def squad_f1(prediction, gold_answers):
    pred_tokens = _squad_normalize(prediction).split()
    best = 0.0
    for g in gold_answers:
        gold_tokens = _squad_normalize(g).split()
        if len(pred_tokens) == 0 or len(gold_tokens) == 0:
            best = max(best, float(pred_tokens == gold_tokens))
            continue
        common = Counter(pred_tokens) & Counter(gold_tokens)
        num_same = sum(common.values())
        if num_same == 0:
            continue
        precision = num_same / len(pred_tokens)
        recall = num_same / len(gold_tokens)
        best = max(best, 2 * precision * recall / (precision + recall))
    return best


In [ ]:
# Block - SQuAD1.1 driver: hand-rolled prefill + greedy-decode loop through
# the official H2O engine, identical structure to generate_ruler_h2o/
# generate_gsm8k_h2o (make_h2o_engines/h2o_step, eviction happening INSIDE
# the timed bracket, same TTFT/TBT/latency definitions), scoring a short
# generated answer per question against SQuAD's official Exact Match / F1
# metrics (best-of however many gold reference answers a question has).


@torch.no_grad()
def generate_squad_h2o(prompt):
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    engines = make_h2o_engines(prompt_len)

    sync_if_cuda()
    gen_start = time.perf_counter()

    outputs = model_h2o(input_ids=enc["input_ids"], use_cache=True,
                        output_attentions=True, return_dict=True)
    last_logits = outputs.logits[:, -1, :]
    next_id = int(last_logits.argmax(dim=-1)[0].item())
    sync_if_cuda()
    ttft_sec = time.perf_counter() - gen_start

    pkv = cache_to_legacy(outputs.past_key_values)
    pkv = apply_engines(pkv, outputs.attentions, engines)

    max_cache_tokens = get_cache_tokens(pkv)
    generated_ids = []
    generated_logits = []
    n_forward_tokens = 1

    for step in range(SQUAD_MAX_NEW_TOKENS):
        if next_id == tokenizer.eos_token_id:
            break
        generated_ids.append(next_id)
        generated_logits.append(last_logits.detach())
        if "\n" in tokenizer.decode([next_id]):
            break

        if step == SQUAD_MAX_NEW_TOKENS - 1:
            break

        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        abs_pos = prompt_len + step
        last_logits, pkv = h2o_step(token_tensor, abs_pos, pkv, engines)
        next_id = int(last_logits.argmax(dim=-1)[0].item())
        n_forward_tokens += 1

        cache_tokens = get_cache_tokens(pkv)
        max_cache_tokens = max(max_cache_tokens, cache_tokens)

    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    if generated_ids:
        nll_sum = 0.0
        for tok_id, logits in zip(generated_ids, generated_logits):
            log_probs = torch.log_softmax(logits[0].float(), dim=-1)
            nll_sum += -log_probs[tok_id].item()
        perplexity = math.exp(min(nll_sum / len(generated_ids), 50.0))
    else:
        perplexity = None

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    n_generated = n_forward_tokens
    if len(generated_ids) == 0:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated
    bpt = kv_bytes_per_token(pkv)
    peak_bytes = int(round(max_cache_tokens * bpt))

    return {
        "prefill_tokens": prompt_len, "generated_tokens": len(generated_ids), "gen_text": gen_text,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec, "total_latency_sec": total_latency_sec,
        "total_tokens": total_tokens, "peak_memory_bytes": peak_bytes, "perplexity": perplexity,
    }


def evaluate_squad_h2o(items, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values, f1_values = [], [], [], [], [], []
    per_question_records = []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, item in enumerate(tqdm(items, desc=f"SQuAD1.1 | {method_label}")):
        prompt = format_squad_prompt(item)
        result = generate_squad_h2o(prompt)
        prediction = result["gen_text"].strip()
        em = squad_exact_match(prediction, item["gold_answers"])
        f1 = squad_f1(prediction, item["gold_answers"])
        correct += em
        total += 1
        f1_values.append(f1)

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- SQuAD1.1 | {method_label} | question {q_idx} preview ---")
            print(f"Question:     {item['question']}")
            print(f"Gold answers: {item['gold_answers']}")
            print(f"Generated:    {prediction!r}")
            print(f"EM: {em} | F1: {f1:.3f}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        if result["perplexity"] is not None:
            ppl_values.append(result["perplexity"])

        per_question_records.append({
            "question_index": q_idx,
            "full_prompt": prompt,
            "prefill_tokens": result["prefill_tokens"],
            "generated_output": result["gen_text"],
            "generated_tokens": result["generated_tokens"],
            "gold_answer": " / ".join(item["gold_answers"]),
            "predicted_answer": prediction,
            "correct": em,
            "f1": f1,
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_f1 = sum(f1_values) / len(f1_values) if f1_values else float("nan")
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_squad_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_question_records)} per-question SQuAD1.1 rows to {_per_prompt_path}")

    return {
        "dataset": "SQuAD1.1",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "avg_f1": avg_f1,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


squad_results = [
    evaluate_squad_h2o(squad_items, METHOD_NAME),
]
squad_results_df = pd.DataFrame(squad_results)
display(squad_results_df)


## Save Results

In [ ]:
# Block - Save combined results (GSM8K + ARC-Challenge + HellaSwag) to CSV --
# identical column schema to the KVQuant-family notebooks, saved into the
# same Drive folder, so all methods' CSVs concatenate directly into one
# comparison table.
#
# Cross-method comparison reminders:
#   - This notebook's memory numbers are MEASURED (physically pruned cache);
#     the quantized KVQuant simulation notebook's are CALCULATED (simulated
#     compression); the REAL packed KVQuant notebook's are also MEASURED.
#     Say so when presenting them side by side.
#   - The full-precision baseline notebook now also runs eager attention
#     with the same hand-rolled prefill/decode loop structure as this
#     notebook (see its Setup section), so there is no longer an
#     "eager tax" confound in the latency columns between this notebook and
#     the baseline; any remaining gap is attributable to H2O's eviction
#     overhead.
#   - For cross-method comparison, prefer average_memory_mb over
#     peak_memory_mb as the primary memory axis. peak_memory_mb is the max
#     over 2,048 questions and is therefore sensitive to which few
#     questions happened to land in the seeded subset with the longest
#     generations (GSM8K especially, where generation length varies widely
#     up to GSM8K_MAX_NEW_TOKENS), not purely a property of the eviction
#     budget. Now that GSM8K's budget is sized from prompt length alone
#     (see generate_gsm8k_h2o), peak/average should behave consistently
#     with ARC-Challenge/HellaSwag, but average remains the more robust
#     axis.
#   - Confirm the GPU name printed in Block 2 matches every other notebook's
#     run before trusting any timing/memory comparison.
#
# Robust to partial runs: only concatenates whichever of gsm8k_results_df /
# arc_results_df / hellaswag_results_df / ruler_results_df actually exist in
# this session, so you can run just a subset of datasets' cells without this
# cell crashing on a NameError for a dataframe you never created -- RULER in
# particular may not always finish (16K-token eager attention is memory-
# heavy), so this matters more here than it did before RULER existed.

_result_df_names = ["gsm8k_results_df", "arc_results_df", "hellaswag_results_df", "ruler_results_df", "squad_results_df"]
_available_dfs = []
for _name in _result_df_names:
    if _name in globals():
        _available_dfs.append(globals()[_name])
    else:
        print(f"Note: {_name} not found in this session -- skipping (its dataset's cells were not run).")

results_df = pd.concat(_available_dfs, ignore_index=True)
results_df = results_df[[
    "dataset", "method", "perplexity", "accuracy",
    "ttft_sec", "tbt_sec", "avg_total_latency_sec",
    "peak_memory_mb", "average_memory_mb",
]]
display(results_df)

os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)

_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{METHOD_NAME}_results.csv"
results_df.to_csv(_path, index=False)
print(f"Saved to {_path}")
